# Analise de Dados BCB

Notebook para analises dos dados coletados do Banco Central do Brasil e outras fontes.

**Analises:**
1. Estatisticas Descritivas Avancadas
2. Analise de Correlacoes
3. Retornos e Variacoes
4. Expectativas vs Realizado
5. Decomposicao Sazonal
6. Resumo e Insights
7. Analise IPEA - Mercado de Trabalho
8. Analise MTE CAGED - Microdados

## 1. Setup e Carregamento

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# MTE CAGED
from mte.caged import CAGEDCollector

# Configurar visualizacao
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10
sns.set_palette('husl')

# Caminho para dados
data_path = Path.cwd().parent / 'data'
processed_path = data_path / 'processed'

# Inicializar collectors
caged_collector = CAGEDCollector(data_path)

print(f"Data path: {data_path}")

In [ ]:
# Carregar dados consolidados
df_daily = pd.read_parquet(processed_path / 'bacen_sgs_daily_consolidated.parquet')
df_monthly = pd.read_parquet(processed_path / 'bacen_sgs_monthly_consolidated.parquet')
df_exp = pd.read_parquet(processed_path / 'expectations_consolidated.parquet')

print("Dados carregados:")
print(f"  SGS Daily: {df_daily.shape[0]:,} registros, {list(df_daily.columns)}")
print(f"  SGS Monthly: {df_monthly.shape[0]:,} registros, {list(df_monthly.columns)}")
print(f"  Expectations: {df_exp.shape[0]:,} registros")

## 2. Estatisticas Descritivas Avancadas

### 2.1 Resumo Estatistico

In [ ]:
# Estatisticas descritivas - SGS Daily
print("=" * 60)
print("SGS DAILY - Estatisticas Descritivas")
print("=" * 60)
display(df_daily.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(4))

In [ ]:
# Estatisticas descritivas - SGS Monthly
print("=" * 60)
print("SGS MONTHLY - Estatisticas Descritivas")
print("=" * 60)
display(df_monthly.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(4))

### 2.2 Analise de Volatilidade

In [ ]:
# Volatilidade rolling (desvio padrao movel) - Cambio
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Dolar PTAX
if 'dolar_ptax' in df_daily.columns:
    # Filtrando dados a partir de 2000
    dolar = df_daily['dolar_ptax'].dropna()
    dolar = dolar[dolar.index >= '2000-01-01']
    
    # Retorno percentual diario
    dolar_ret = dolar.pct_change() * 100
    # Volatilidade rolling 21 dias (1 mes)
    dolar_vol = dolar_ret.rolling(window=21).std()
    
    axes[0].plot(dolar.index, dolar, linewidth=0.8, label='Dolar PTAX')
    axes[0].set_title('Dolar PTAX - Cotacao (a partir de 2000)')
    axes[0].set_ylabel('R$')
    axes[0].legend()
    
    axes[1].fill_between(dolar_vol.index, 0, dolar_vol, alpha=0.5, label='Volatilidade 21d')
    axes[1].set_title('Dolar PTAX - Volatilidade (Desvio Padrao Rolling 21 dias)')
    axes[1].set_ylabel('Volatilidade (%)')
    axes[1].set_xlabel('Data')
    axes[1].legend()

plt.tight_layout()
plt.show()

# Estatisticas de volatilidade
if 'dolar_ptax' in df_daily.columns:
    print(f"\nVolatilidade Media (21d): {dolar_vol.mean():.4f}%")
    print(f"Volatilidade Max (21d): {dolar_vol.max():.4f}%")
    print(f"Data de maior volatilidade: {dolar_vol.idxmax().strftime('%Y-%m-%d')}")

In [ ]:
# Volatilidade rolling - Taxa Selic
if 'selic' in df_daily.columns:
    selic = df_daily['selic'].dropna()
    selic_vol = selic.rolling(window=63).std()  # ~3 meses
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    axes[0].plot(selic.index, selic, linewidth=0.8, color='darkblue')
    axes[0].set_title('Meta Selic - Taxa')
    axes[0].set_ylabel('% a.a.')
    
    axes[1].fill_between(selic_vol.index, 0, selic_vol, alpha=0.5, color='darkblue')
    axes[1].set_title('Meta Selic - Volatilidade (Desvio Padrao Rolling 63 dias)')
    axes[1].set_ylabel('Volatilidade (p.p.)')
    axes[1].set_xlabel('Data')
    
    plt.tight_layout()
    plt.show()

### 2.3 Tendencias (Medias Moveis)

In [ ]:
# Medias moveis - Dolar
if 'dolar_ptax' in df_daily.columns:
    dolar = df_daily['dolar_ptax'].dropna()
    dolar = dolar[dolar.index >= '2000-01-01']
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    ax.plot(dolar.index, dolar, linewidth=0.5, alpha=0.7, label='Dolar PTAX')
    ax.plot(dolar.index, dolar.rolling(21).mean(), linewidth=1.5, label='MM 21d (1 mes)')
    ax.plot(dolar.index, dolar.rolling(63).mean(), linewidth=1.5, label='MM 63d (3 meses)')
    ax.plot(dolar.index, dolar.rolling(252).mean(), linewidth=2, label='MM 252d (1 ano)')
    
    ax.set_title('Dolar PTAX - Medias Moveis')
    ax.set_ylabel('R$')
    ax.set_xlabel('Data')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Medias moveis - IBC-Br
if 'ibc_br_dessaz' in df_monthly.columns:
    ibc = df_monthly['ibc_br_dessaz'].dropna()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    ax.plot(ibc.index, ibc, linewidth=1, alpha=0.7, label='IBC-Br Dessaz', marker='o', markersize=2)
    ax.plot(ibc.index, ibc.rolling(3).mean(), linewidth=1.5, label='MM 3 meses')
    ax.plot(ibc.index, ibc.rolling(12).mean(), linewidth=2, label='MM 12 meses')
    
    ax.set_title('IBC-Br Dessazonalizado - Medias Moveis')
    ax.set_ylabel('Indice')
    ax.set_xlabel('Data')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

### 2.4 Decomposicao Sazonal (Series Mensais)

In [ ]:
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    
    if 'igp_m' in df_monthly.columns:
        # Filter data from year 2000 onwards
        igpm = df_monthly['igp_m'].dropna()
        igpm = igpm[igpm.index >= '2000-01-01']
        
        # Decomposicao aditiva
        decomposition = seasonal_decompose(igpm, model='additive', period=12)
        
        fig, axes = plt.subplots(4, 1, figsize=(14, 12))
        
        decomposition.observed.plot(ax=axes[0], title='IGP-M - Serie Original (desde 2000)')
        decomposition.trend.plot(ax=axes[1], title='Tendencia')
        decomposition.seasonal.plot(ax=axes[2], title='Sazonalidade')
        decomposition.resid.plot(ax=axes[3], title='Residuo')
        
        for ax in axes:
            ax.set_xlabel('')
        axes[3].set_xlabel('Data')
        
        plt.tight_layout()
        plt.show()
        
except ImportError:
    print("statsmodels nao instalado. Instale com: pip install statsmodels")

## 3. Analise de Correlacoes

### 3.1 Matriz de Correlacao - SGS Daily

In [ ]:
# Matriz de correlacao - SGS Daily
corr_daily = df_daily.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_daily, dtype=bool))
sns.heatmap(corr_daily, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.3f', square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Matriz de Correlacao - SGS Daily')
plt.tight_layout()
plt.show()

print("\nCorrelacoes:")
display(corr_daily.round(4))

### 3.2 Matriz de Correlacao - SGS Monthly

In [ ]:
# Matriz de correlacao - SGS Monthly
corr_monthly = df_monthly.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_monthly, dtype=bool))
sns.heatmap(corr_monthly, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.3f', square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Matriz de Correlacao - SGS Monthly')
plt.tight_layout()
plt.show()

print("\nCorrelacoes:")
display(corr_monthly.round(4))

### 3.3 Correlacao Cruzada com Defasagem (Lag Correlation)

In [ ]:
def lag_correlation(series1, series2, max_lag=30):
    """Calcula correlacao entre series com diferentes defasagens."""
    correlations = []
    lags = range(-max_lag, max_lag + 1)
    
    for lag in lags:
        if lag < 0:
            corr = series1.shift(-lag).corr(series2)
        else:
            corr = series1.corr(series2.shift(lag))
        correlations.append(corr)
    
    return pd.Series(correlations, index=lags)

# Correlacao defasada: Selic vs Dolar
if 'selic' in df_daily.columns and 'dolar_ptax' in df_daily.columns:
    selic = df_daily['selic'].dropna()
    dolar = df_daily['dolar_ptax'].dropna()
    
    # Alinhar indices
    common_idx = selic.index.intersection(dolar.index)
    selic_aligned = selic.loc[common_idx]
    dolar_aligned = dolar.loc[common_idx]
    
    lag_corr = lag_correlation(selic_aligned, dolar_aligned, max_lag=60)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(lag_corr.index, lag_corr.values, alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title('Correlacao Defasada: Selic vs Dolar PTAX')
    ax.set_xlabel('Lag (dias) - Negativo: Selic lidera | Positivo: Dolar lidera')
    ax.set_ylabel('Correlacao')
    plt.tight_layout()
    plt.show()
    
    # Lag com maior correlacao
    best_lag = lag_corr.abs().idxmax()
    print(f"\nMaior correlacao (em modulo): {lag_corr[best_lag]:.4f} com lag de {best_lag} dias")

In [ ]:
# Correlacao defasada: IBC-Br vs IGP-M (mensal)
if 'ibc_br_dessaz' in df_monthly.columns and 'igp_m' in df_monthly.columns:
    ibc = df_monthly['ibc_br_dessaz'].dropna()
    igpm = df_monthly['igp_m'].dropna()
    
    common_idx = ibc.index.intersection(igpm.index)
    ibc_aligned = ibc.loc[common_idx]
    igpm_aligned = igpm.loc[common_idx]
    
    lag_corr = lag_correlation(ibc_aligned, igpm_aligned, max_lag=12)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(lag_corr.index, lag_corr.values, alpha=0.7, color='green')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title('Correlacao Defasada: IBC-Br Dessaz vs IGP-M')
    ax.set_xlabel('Lag (meses) - Negativo: IBC-Br lidera | Positivo: IGP-M lidera')
    ax.set_ylabel('Correlacao')
    plt.tight_layout()
    plt.show()
    
    best_lag = lag_corr.abs().idxmax()
    print(f"\nMaior correlacao (em modulo): {lag_corr[best_lag]:.4f} com lag de {best_lag} meses")

## 4. Retornos e Variacoes

### 4.1 Variacoes Percentuais

In [ ]:
# Calcular variacoes percentuais diarias
df_daily_ret = df_daily.pct_change(fill_method=None) * 100

print("Variacoes Percentuais Diarias (%)")
print("=" * 60)
display(df_daily_ret.describe().round(4))

In [ ]:
# Histograma de retornos - Dolar
if 'dolar_ptax' in df_daily_ret.columns:
    # Filter data from 2000 onwards
    dolar_ret = df_daily_ret.loc['2000':, 'dolar_ptax'].dropna()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histograma
    axes[0].hist(dolar_ret, bins=100, density=True, alpha=0.7, edgecolor='black')
    
    # Overlay distribuicao normal
    x = np.linspace(dolar_ret.min(), dolar_ret.max(), 100)
    axes[0].plot(x, stats.norm.pdf(x, dolar_ret.mean(), dolar_ret.std()), 'r-', linewidth=2, label='Normal')
    
    axes[0].set_title('Distribuicao dos Retornos Diarios - Dolar PTAX (desde 2000)')
    axes[0].set_xlabel('Retorno (%)')
    axes[0].set_ylabel('Densidade')
    axes[0].legend()
    
    # QQ-Plot
    stats.probplot(dolar_ret, dist="norm", plot=axes[1])
    axes[1].set_title('QQ-Plot - Dolar PTAX (desde 2000)')
    
    plt.tight_layout()
    plt.show()
    
    # Estatisticas de distribuicao
    print(f"\nEstatisticas de Distribuicao (desde 2000):")
    print(f"  Assimetria (Skewness): {stats.skew(dolar_ret):.4f}")
    print(f"  Curtose: {stats.kurtosis(dolar_ret):.4f}")
    print(f"  Teste Jarque-Bera (normalidade): {stats.jarque_bera(dolar_ret)}")

### 4.2 Retornos Acumulados

In [ ]:
# Retornos acumulados - Cambio
fig, ax = plt.subplots(figsize=(14, 6))

# Filter data from 2000 onwards
df_filtered = df_daily[df_daily.index >= '2000-01-01']

for col in ['dolar_ptax', 'euro_ptax']:
    if col in df_filtered.columns:
        serie = df_filtered[col].dropna()
        if not serie.empty:
            # Normalizar para base 100
            serie_norm = (serie / serie.iloc[0]) * 100
            ax.plot(serie_norm.index, serie_norm, linewidth=1, label=col)

ax.axhline(y=100, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_title('Evolucao do Cambio (Base 100) - Desde 2000')
ax.set_xlabel('Data')
ax.set_ylabel('Indice (Base 100)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Retornos acumulados por ano
if 'dolar_ptax' in df_daily.columns:
    dolar = df_daily['dolar_ptax'].dropna()
    
    # Limit analysis from 2000 onwards
    dolar = dolar[dolar.index.year >= 2000]
    
    # Retorno anual
    retorno_anual = dolar.resample('YE').last().pct_change() * 100
    retorno_anual = retorno_anual.dropna()
    retorno_anual.index = retorno_anual.index.year
    
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['green' if x > 0 else 'red' for x in retorno_anual.values]
    ax.bar(retorno_anual.index, retorno_anual.values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_title('Retorno Anual do Dolar PTAX')
    ax.set_xlabel('Ano')
    ax.set_ylabel('Retorno (%)')
    
    # Adicionar valores nas barras
    for i, (ano, ret) in enumerate(retorno_anual.items()):
        ax.text(ano, ret + (2 if ret > 0 else -4), f'{ret:.1f}%', ha='center', fontsize=8)
    
    plt.tight_layout()
    plt.show()

### 4.3 Drawdown Analysis

In [ ]:
def calculate_drawdown(series):
    """Calcula drawdown de uma serie de precos."""
    # Maximo acumulado
    rolling_max = series.expanding().max()
    # Drawdown percentual
    drawdown = (series - rolling_max) / rolling_max * 100
    return drawdown

# Drawdown do Dolar (queda em relacao ao pico)
if 'dolar_ptax' in df_daily.columns:
    dolar = df_daily['dolar_ptax'].dropna()
    # Limit analysis from 2000 onwards
    dolar = dolar[dolar.index >= '2000-01-01']
    
    dolar_dd = calculate_drawdown(dolar)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Preco
    axes[0].plot(dolar.index, dolar, linewidth=0.8)
    axes[0].set_title('Dolar PTAX - Cotacao (A partir de 2000)')
    axes[0].set_ylabel('R$')
    
    # Drawdown
    axes[1].fill_between(dolar_dd.index, 0, dolar_dd, color='red', alpha=0.5)
    axes[1].set_title('Drawdown (Queda em relacao ao pico historico) (A partir de 2000)')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Data')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMaior Drawdown: {dolar_dd.min():.2f}%")
    print(f"Data do maior drawdown: {dolar_dd.idxmin().strftime('%Y-%m-%d')}")

In [ ]:
# Drawdown do IBC-Br (atividade economica)
if 'ibc_br_dessaz' in df_monthly.columns:
    ibc = df_monthly['ibc_br_dessaz'].dropna()
    ibc_dd = calculate_drawdown(ibc)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    axes[0].plot(ibc.index, ibc, linewidth=1, marker='o', markersize=2)
    axes[0].set_title('IBC-Br Dessazonalizado - Indice')
    axes[0].set_ylabel('Indice')
    
    axes[1].fill_between(ibc_dd.index, 0, ibc_dd, color='red', alpha=0.5)
    axes[1].set_title('Drawdown - Queda da Atividade Economica')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Data')
    
    plt.tight_layout()
    plt.show()
    
    # Top 5 maiores drawdowns
    print("\nTop 5 maiores quedas da atividade economica:")
    top_dd = ibc_dd.nsmallest(5)
    for date, dd in top_dd.items():
        print(f"  {date.strftime('%Y-%m')}: {dd:.2f}%")

## 5. Expectativas vs Realizado

### 5.1 Preparacao dos Dados

In [ ]:
# Verificar estrutura das expectations
print("Colunas disponveis em Expectations:")
print(df_exp.columns.tolist())
print(f"\nIndicadores disponiveis (_source):")
print(df_exp['_source'].unique())

In [ ]:
# Funcao para converter DataReferencia para datetime
def parse_data_referencia(df):
    """Converte coluna DataReferencia (MM/YYYY) para datetime."""
    df = df.copy()
    
    # Preencher DataReferencia com Reuniao se estiver vazio (para Selic)
    if 'Reuniao' in df.columns and 'DataReferencia' in df.columns:
        df['DataReferencia'] = df['DataReferencia'].fillna(df['Reuniao'])
    
    if 'DataReferencia' in df.columns:
        # Tratar diferentes formatos
        def convert_ref(val):
            if pd.isna(val):
                return pd.NaT
            try:
                val_str = str(val)
                # Formato MM/YYYY ou Rn/YYYY
                if '/' in val_str:
                    parts = val_str.split('/')
                    if len(parts) == 2:
                        # Verificar se e reuniao (R1/2024)
                        if parts[0].upper().startswith('R'):
                            # Retorna 01/01/ANO para permitir filtro por ano
                            return pd.Timestamp(year=int(parts[1]), month=1, day=1)
                        
                        month, year = parts
                        return pd.Timestamp(year=int(year), month=int(month), day=1)
                # Formato YYYY
                elif len(val_str) == 4:
                    return pd.Timestamp(year=int(val_str), month=12, day=31)
            except Exception:
                pass
            return pd.NaT
        
        df['DataReferencia_dt'] = df['DataReferencia'].apply(convert_ref)
    return df

df_exp_parsed = parse_data_referencia(df_exp)
print("Amostra com DataReferencia convertida:")
display(df_exp_parsed[['Indicador', 'Mediana', '_source', 'DataReferencia', 'DataReferencia_dt']].head(10))

# Verificar Selic especificamente para garantir que funcionou
if 'selic' in df_exp_parsed['_source'].unique():
    print("\nAmostra Selic (com DataReferencia preenchida):")
    display(df_exp_parsed[df_exp_parsed['_source'] == 'selic'][['Indicador', 'Mediana', '_source', 'DataReferencia', 'DataReferencia_dt']].head(5))

### 5.2 Evolucao das Expectativas ao Longo do Tempo

In [ ]:
# Exemplo: Evolucao das expectativas de Selic para um ano especifico
if 'selic' in df_exp_parsed['_source'].values:
    selic_exp = df_exp_parsed[df_exp_parsed['_source'] == 'selic'].copy()
    
    # Pegar expectativas para reunioes de 2024
    if 'DataReferencia_dt' in selic_exp.columns:
        # Filtrar expectativas para 2024
        selic_2024 = selic_exp[
            (selic_exp['DataReferencia_dt'].dt.year == 2024) & 
            (selic_exp['DataReferencia_dt'].notna())
        ].copy()
        
        if len(selic_2024) > 0:
            fig, ax = plt.subplots(figsize=(14, 6))
            
            # Agrupar por data da pesquisa e DataReferencia, calcular mediana
            for ref_date in selic_2024['DataReferencia'].unique()[:6]:  # Primeiras 6 reunioes
                subset = selic_2024[selic_2024['DataReferencia'] == ref_date]
                mediana_serie = subset.groupby(subset.index)['Mediana'].mean()
                ax.plot(mediana_serie.index, mediana_serie, label=f'Ref: {ref_date}', linewidth=1)
            
            ax.set_title('Evolucao das Expectativas de Selic - Reunioes 2024')
            ax.set_xlabel('Data da Pesquisa')
            ax.set_ylabel('Expectativa Selic (%)')
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()
        else:
            print("Nao ha dados de Selic para 2024")

In [ ]:
# Evolucao das expectativas de IPCA anual
if 'ipca_anual' in df_exp_parsed['_source'].values:
    ipca_exp = df_exp_parsed[df_exp_parsed['_source'] == 'ipca_anual'].copy()
    
    if 'DataReferencia_dt' in ipca_exp.columns:
        # Pegar anos recentes
        anos_ref = [2023, 2024, 2025]
        
        fig, ax = plt.subplots(figsize=(14, 6))
        
        for ano in anos_ref:
            subset = ipca_exp[ipca_exp['DataReferencia_dt'].dt.year == ano]
            if len(subset) > 0:
                mediana_serie = subset.groupby(subset.index)['Mediana'].mean()
                ax.plot(mediana_serie.index, mediana_serie, label=f'IPCA {ano}', linewidth=1)
        
        ax.set_title('Evolucao das Expectativas de IPCA Anual')
        ax.set_xlabel('Data da Pesquisa')
        ax.set_ylabel('Expectativa IPCA (%)')
        ax.legend()
        plt.tight_layout()
        plt.show()

### 5.3 Dispersao das Expectativas

In [ ]:
# Analise da dispersao (desvio padrao) das expectativas ao longo do tempo
if 'ipca_anual' in df_exp_parsed['_source'].values:
    ipca_exp = df_exp_parsed[df_exp_parsed['_source'] == 'ipca_anual'].copy()
    
    if 'DesvioPadrao' in ipca_exp.columns:
        # Media do desvio padrao por data
        dispersao = ipca_exp.groupby(ipca_exp.index)['DesvioPadrao'].mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.fill_between(dispersao.index, 0, dispersao, alpha=0.5)
        ax.plot(dispersao.index, dispersao, linewidth=0.5)
        ax.set_title('Dispersao das Expectativas de IPCA (Desvio Padrao)')
        ax.set_xlabel('Data')
        ax.set_ylabel('Desvio Padrao')
        plt.tight_layout()
        plt.show()
        
        # Periodos de maior incerteza
        print("\nPeriodos de maior incerteza (top 10):")
        top_dispersao = dispersao.nlargest(10)
        for date, val in top_dispersao.items():
            print(f"  {date.strftime('%Y-%m-%d')}: {val:.4f}")

### 5.4 Erro de Previsao

In [ ]:
# Comparar expectativas de cambio com valor realizado
# Nota: Esta analise requer dados de IPCA realizado que nao temos no SGS atual
# Vamos usar IGP-M como proxy para demonstrar a metodologia

if 'igpm_mensal' in df_exp_parsed['_source'].values and 'igp_m' in df_monthly.columns:
    # Expectativas de IGP-M mensal
    igpm_exp = df_exp_parsed[df_exp_parsed['_source'] == 'igpm_mensal'].copy()
    
    # IGP-M realizado
    igpm_real = df_monthly['igp_m'].dropna()
    
    print("Comparando expectativas de IGP-M com valores realizados...")
    print(f"Expectativas: {len(igpm_exp)} registros")
    print(f"Realizado: {len(igpm_real)} registros")
    
    # Pegar ultima expectativa antes de cada mes realizado
    erros = []
    
    for date, valor_real in igpm_real.items():
        # Expectativas para este mes de referencia
        ref_str = f"{date.month:02d}/{date.year}"
        exp_mes = igpm_exp[
            (igpm_exp['DataReferencia'] == ref_str) &
            (igpm_exp.index < date)
        ]
        
        if len(exp_mes) > 0:
            # Ultima expectativa antes da realizacao
            ultima_exp = exp_mes.iloc[-1]
            if pd.notna(ultima_exp['Mediana']):
                erro = ultima_exp['Mediana'] - valor_real
                erros.append({
                    'data_ref': date,
                    'expectativa': ultima_exp['Mediana'],
                    'realizado': valor_real,
                    'erro': erro,
                    'erro_abs': abs(erro)
                })
    
    if erros:
        df_erros = pd.DataFrame(erros)
        df_erros.set_index('data_ref', inplace=True)
        
        # Metricas de erro
        mae = df_erros['erro_abs'].mean()
        rmse = np.sqrt((df_erros['erro'] ** 2).mean())
        me = df_erros['erro'].mean()  # Erro medio (vies)
        
        print(f"\nMetricas de Erro de Previsao (IGP-M):")
        print(f"  MAE (Erro Absoluto Medio): {mae:.4f}")
        print(f"  RMSE (Raiz do Erro Quadratico Medio): {rmse:.4f}")
        print(f"  ME (Erro Medio - Vies): {me:.4f}")
        
        # Grafico
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        
        # Expectativa vs Realizado
        axes[0].plot(df_erros.index, df_erros['expectativa'], label='Expectativa', linewidth=1)
        axes[0].plot(df_erros.index, df_erros['realizado'], label='Realizado', linewidth=1)
        axes[0].set_title('IGP-M: Expectativa vs Realizado')
        axes[0].set_ylabel('IGP-M (%)')
        axes[0].legend()
        
        # Erro
        colors = ['green' if x < 0 else 'red' for x in df_erros['erro']]
        axes[1].bar(df_erros.index, df_erros['erro'], color=colors, alpha=0.7, width=20)
        axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        axes[1].set_title('Erro de Previsao (Expectativa - Realizado)')
        axes[1].set_ylabel('Erro (p.p.)')
        axes[1].set_xlabel('Data de Referencia')
        
        plt.tight_layout()
        plt.show()
    else:
        print("Nao foi possivel calcular erros - verifique o formato de DataReferencia")
else:
    print("Dados de IGP-M (expectativa ou realizado) nao disponiveis para comparacao")

## 6. Resumo e Insights

In [ ]:
# Filter data from year 2000 forward
df_daily = df_daily[df_daily.index >= '2000-01-01']
df_monthly = df_monthly[df_monthly.index >= '2000-01-01']
df_exp = df_exp[df_exp.index >= '2000-01-01']

print("=" * 70)
print("RESUMO DAS ANALISES (DADOS A PARTIR DE 2000)")
print("=" * 70)

print("\n1. ESTATISTICAS DESCRITIVAS")
print("-" * 40)
if 'dolar_ptax' in df_daily.columns:
    dolar = df_daily['dolar_ptax'].dropna()
    if not dolar.empty:
        print(f"   Dolar PTAX: R$ {dolar.iloc[-1]:.4f} (atual)")
        print(f"   Variacao no periodo: {((dolar.iloc[-1]/dolar.iloc[0])-1)*100:.2f}%")

if 'selic' in df_daily.columns:
    selic = df_daily['selic'].dropna()
    if not selic.empty:
        print(f"   Selic atual: {selic.iloc[-1]:.2f}% a.a.")

print("\n2. CORRELACOES PRINCIPAIS")
print("-" * 40)
if 'selic' in df_daily.columns and 'dolar_ptax' in df_daily.columns:
    corr = df_daily['selic'].corr(df_daily['dolar_ptax'])
    print(f"   Selic x Dolar: {corr:.4f}")

if 'dolar_ptax' in df_daily.columns and 'euro_ptax' in df_daily.columns:
    corr = df_daily['dolar_ptax'].corr(df_daily['euro_ptax'])
    print(f"   Dolar x Euro: {corr:.4f}")

print("\n3. DADOS DISPONIVEIS")
print("-" * 40)
if not df_daily.empty:
    print(f"   SGS Daily: {df_daily.index.min().strftime('%Y-%m-%d')} a {df_daily.index.max().strftime('%Y-%m-%d')}")
if not df_monthly.empty:
    print(f"   SGS Monthly: {df_monthly.index.min().strftime('%Y-%m-%d')} a {df_monthly.index.max().strftime('%Y-%m-%d')}")
if not df_exp.empty:
    print(f"   Expectations: {df_exp.index.min().strftime('%Y-%m-%d')} a {df_exp.index.max().strftime('%Y-%m-%d')}")

print("\n" + "=" * 70)

## 7. Analise IPEA - Mercado de Trabalho

In [ ]:
# Carregar dados IPEA
df_ipea = pd.read_parquet(processed_path / 'ipea_monthly_consolidated.parquet')

print("IPEA - Dados Agregados de Emprego")
print(f"Periodo: {df_ipea.index.min().strftime('%Y-%m')} a {df_ipea.index.max().strftime('%Y-%m')}")
print(f"Colunas: {list(df_ipea.columns)}")

In [ ]:
print("=" * 60)
print("IPEA - Estatisticas Descritivas")
print("=" * 60)
display(df_ipea.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2))

In [ ]:
# Correlacao entre saldo CAGED e taxa de desemprego
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Serie temporal
ax1 = axes[0]
ax2 = ax1.twinx()

ax1.plot(df_ipea.index, df_ipea['caged_saldo'], 'b-', linewidth=1, label='Saldo CAGED')
ax2.plot(df_ipea.index, df_ipea['taxa_desemprego'], 'r-', linewidth=1, label='Taxa Desemprego')

ax1.set_xlabel('Data')
ax1.set_ylabel('Saldo CAGED', color='b')
ax2.set_ylabel('Taxa Desemprego (%)', color='r')
ax1.axhline(y=0, color='blue', linestyle='--', alpha=0.3)
axes[0].set_title('Saldo CAGED vs Taxa de Desemprego')

# Scatter
axes[1].scatter(df_ipea['caged_saldo'], df_ipea['taxa_desemprego'], alpha=0.5)
axes[1].set_xlabel('Saldo CAGED')
axes[1].set_ylabel('Taxa Desemprego (%)')
corr = df_ipea['caged_saldo'].corr(df_ipea['taxa_desemprego'])
axes[1].set_title(f'Correlacao: {corr:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

saldo = df_ipea['caged_saldo']
ax.plot(saldo.index, saldo, linewidth=0.5, alpha=0.7, label='Saldo Mensal')
ax.plot(saldo.index, saldo.rolling(3).mean(), linewidth=1.5, label='MM 3 meses')
ax.plot(saldo.index, saldo.rolling(12).mean(), linewidth=2, label='MM 12 meses')

ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_title('Saldo CAGED - Tendencia (Medias Moveis)')
ax.set_xlabel('Data')
ax.set_ylabel('Saldo')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# Saldo medio por mes do ano
saldo_mensal = df_ipea['caged_saldo'].groupby(df_ipea.index.month).mean()

fig, ax = plt.subplots(figsize=(12, 5))

colors = ['green' if x > 0 else 'red' for x in saldo_mensal]
bars = ax.bar(range(1, 13), saldo_mensal, color=colors, edgecolor='black')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                    'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Saldo CAGED - Sazonalidade (Media por Mes)')
ax.set_xlabel('Mes')
ax.set_ylabel('Saldo Medio')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# Saldo anual
saldo_anual = df_ipea['caged_saldo'].resample('YE').sum()

# Filter out years with zero balance (no data)
saldo_anual = saldo_anual[saldo_anual != 0]

saldo_anual.index = saldo_anual.index.year

fig, ax = plt.subplots(figsize=(14, 5))

colors = ['green' if x > 0 else 'red' for x in saldo_anual]
ax.bar(saldo_anual.index, saldo_anual, color=colors, edgecolor='black')

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Saldo CAGED - Acumulado Anual')
ax.set_xlabel('Ano')
ax.set_ylabel('Saldo Anual')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

# Rotulos
for ano, val in saldo_anual.items():
    ax.text(ano, val + (val * 0.05 if val > 0 else val * 0.05 - 100000),
            f'{val/1e6:.1f}M', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Analise MTE CAGED - Microdados

In [ ]:
# Carregar dados de um mes para analise
df_caged = caged_collector.read(year=2025, months=[10])

print("MTE CAGED - Microdados 2025-10")
print(f"Total de movimentacoes: {len(df_caged):,}")
print(f"Admissoes: {(df_caged['saldomovimentação'] == 1).sum():,}")
print(f"Desligamentos: {(df_caged['saldomovimentação'] == -1).sum():,}")
print(f"Saldo: {df_caged['saldomovimentação'].sum():,}")

In [ ]:
# Carregar saldo de multiplos meses para ver evolucao
# Usa get_files() para obter lista de arquivos e carrega apenas coluna necessaria
files = caged_collector.get_files(year=2025)

saldos = []
for f in files:
    df_temp = pd.read_parquet(f, columns=['saldomovimentação'])
    mes = int(f.stem.split('-')[1])
    saldos.append({'mes': mes, 'saldo': df_temp['saldomovimentação'].sum()})
    del df_temp

saldo_mes = pd.DataFrame(saldos).set_index('mes')['saldo']

fig, ax = plt.subplots(figsize=(12, 5))

colors = ['green' if x > 0 else 'red' for x in saldo_mes]
ax.bar(saldo_mes.index, saldo_mes, color=colors, edgecolor='black')

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Saldo CAGED por Mes - 2025')
ax.set_xlabel('Mes')
ax.set_ylabel('Saldo')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao de salarios (remover outliers)
salarios = df_caged['salário'].dropna()
salarios = salarios[(salarios > 0) & (salarios < salarios.quantile(0.99))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(salarios, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=salarios.median(), color='red', linestyle='--', label=f'Mediana: R$ {salarios.median():,.0f}')
axes[0].axvline(x=salarios.mean(), color='blue', linestyle='--', label=f'Media: R$ {salarios.mean():,.0f}')
axes[0].set_title('Distribuicao Salarial (2025-10)')
axes[0].set_xlabel('Salario (R$)')
axes[0].set_ylabel('Frequencia')
axes[0].legend()

# Boxplot por tipo de movimentacao
df_caged_sample = df_caged[df_caged['salário'].between(0, salarios.quantile(0.99))].copy()
df_caged_sample['tipo'] = df_caged_sample['saldomovimentação'].map({1: 'Admissao', -1: 'Desligamento'})
df_caged_sample.boxplot(column='salário', by='tipo', ax=axes[1])
axes[1].set_title('Salario por Tipo de Movimentacao')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Salario (R$)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Salario mediano: R$ {salarios.median():,.2f}")
print(f"Salario medio: R$ {salarios.mean():,.2f}")

In [ ]:
# Saldo por sexo (1=Homem, 3=Mulher conforme layout CAGED)
sexo_map = {1: 'Masculino', 3: 'Feminino'}
df_caged['sexo_nome'] = df_caged['sexo'].map(sexo_map)

sexo_stats = df_caged.groupby('sexo_nome').agg({
    'saldomovimentação': ['sum', 'count'],
    'salário': 'median'
}).round(2)
sexo_stats.columns = ['saldo', 'movimentacoes', 'salario_mediano']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Movimentacoes
axes[0].bar(sexo_stats.index, sexo_stats['movimentacoes'], color=['steelblue', 'coral'])
axes[0].set_title('Movimentacoes por Sexo')
axes[0].set_ylabel('Quantidade')

# Saldo
colors = ['green' if x > 0 else 'red' for x in sexo_stats['saldo']]
axes[1].bar(sexo_stats.index, sexo_stats['saldo'], color=colors)
axes[1].set_title('Saldo por Sexo')
axes[1].set_ylabel('Saldo')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Salario mediano
axes[2].bar(sexo_stats.index, sexo_stats['salario_mediano'], color=['steelblue', 'coral'])
axes[2].set_title('Salario Mediano por Sexo')
axes[2].set_ylabel('Salario (R$)')

plt.tight_layout()
plt.show()

display(sexo_stats)

In [ ]:
# Agregar dados de multiplos meses para heatmap
files = caged_collector.get_files(year=2025)

dfs = []
for f in files:
    df_temp = pd.read_parquet(f, columns=['uf', 'saldomovimentação'])
    mes = int(f.stem.split('-')[1])
    df_temp['mes'] = mes
    dfs.append(df_temp)

df_all = pd.concat(dfs, ignore_index=True)
del dfs

# Pivot: UF x Mes
pivot = df_all.pivot_table(
    values='saldomovimentação',
    index='uf',
    columns='mes',
    aggfunc='sum'
)

# Top 10 UFs por movimentacao total
top_ufs = df_all.groupby('uf')['saldomovimentação'].count().nlargest(10).index
pivot_top = pivot.loc[top_ufs]

fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(pivot_top, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Saldo'})

ax.set_title('Saldo CAGED por UF e Mes - 2025 (Top 10 UFs)')
ax.set_xlabel('Mes')
ax.set_ylabel('UF')

plt.tight_layout()
plt.show()

del df_all

In [ ]:
# Top 10 setores por saldo
secao_stats = df_caged.groupby('seção').agg({
    'saldomovimentação': ['sum', 'count'],
    'salário': 'median'
}).round(2)
secao_stats.columns = ['saldo', 'movimentacoes', 'salario_mediano']
secao_stats = secao_stats.sort_values('saldo', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 5 positivos e top 5 negativos
top_pos = secao_stats.head(5)
top_neg = secao_stats.tail(5)

# Positivos
axes[0].barh(top_pos.index, top_pos['saldo'], color='green')
axes[0].set_title('Top 5 Setores - Maior Saldo Positivo')
axes[0].set_xlabel('Saldo')
axes[0].invert_yaxis()

# Negativos
axes[1].barh(top_neg.index, top_neg['saldo'], color='red')
axes[1].set_title('Top 5 Setores - Maior Saldo Negativo')
axes[1].set_xlabel('Saldo')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Resumo por Secao (Top 10 por saldo):")
display(secao_stats.head(10))